In [ ]:
import time
from io import BytesIO
from PIL import Image, ImageDraw, ImageFont
import torch
from transformers import (
    AutoProcessor,
    AutoModelForZeroShotImageClassification,
    AutoModelForZeroShotObjectDetection
)
import gradio as gr

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
CLASSIFICATION_MODELS = {
    "google/siglip2-base-patch16-224": "google/siglip2-base-patch16-224",
    "openai/clip-vit-large-patch14": "openai/clip-vit-large-patch14",
}

In [ ]:
DETECTION_MODELS = {
    "openmmlab-community/mm_grounding_dino_large_all": "openmmlab-community/mm_grounding_dino_large_all",
    "iSEE-Laboratory/llmdet_large": "iSEE-Laboratory/llmdet_large",
}

In [ ]:
def visualize_prediction(image, detections, threshold=0.3):
  draw = ImageDraw.Draw(image)
  font = ImageFont.load_default(size=30)

  for det in detections:
      score = det.get("score", 0.0)
      if score < threshold:
          continue
      box = det.get("box", None)
      label = det.get("label", "")
      if box is None:
          continue
      xmin, ymin, xmax, ymax = box
      draw.rectangle([xmin, ymin, xmax, ymax], outline="red", width=3)
      text = f"{label} {score:.2f}"
      draw.text((xmin, ymin - 10), text, font=font)
  return image

In [ ]:
def run_classification(model_key, image_file, labels_text):
    img = Image.open(image_file).convert("RGB")
    labels = [x.strip() for x in labels_text.split(",") if x.strip()]

    model_name = CLASSIFICATION_MODELS[model_key]
    processor = AutoProcessor.from_pretrained(model_name)
    model = AutoModelForZeroShotImageClassification.from_pretrained(model_name).to(device)

    start = time.time()

    inputs = processor(text=labels, images=img, return_tensors="pt", padding='max_length').to(device)
    with torch.no_grad():
        outputs = model(**inputs)

    probs = outputs.logits_per_image.softmax(dim=-1)[0]
    scores = probs.tolist()

    elapsed = round(time.time() - start, 4)

    results = list(zip(labels, scores))
    results.sort(key=lambda x: x[1], reverse=True)
    print(results)

    top1 = results[0]
    runner = results[1] if len(results) > 1 else None

    txt = (
        f"Model: {model_name}\n"
        f"Inference time: {elapsed} s\n\n"
        f"Top-1:\n  {top1[0]} ({top1[1]:.4f})\n\n"
        f"Runner-up:\n  {runner[0]} ({runner[1]:.4f})\n\n"
        f"All predictions:\n"
    )

    for label, score in results:
        txt += f"- {label}: {score:.4f}\n"

    return txt

In [ ]:
def run_detection(model_key, image_file, labels_text, threshold):
    img = Image.open(image_file).convert("RGB")
    labels = [x.strip() for x in labels_text.split(",") if x.strip()]

    model_name = DETECTION_MODELS[model_key]
    processor = AutoProcessor.from_pretrained(model_name)
    model = AutoModelForZeroShotObjectDetection.from_pretrained(model_name).to(device)

    start = time.time()

    inputs = processor(text=labels, images=img, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    results = processor.post_process_grounded_object_detection(
        outputs, target_sizes=[img.size[::-1]], threshold=threshold
    )[0]

    elapsed = round(time.time() - start, 4)

    detections = []

    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        detections.append({
            "label": label,
            "score": float(score),
            "box": [float(x) for x in box],
        })

    img_out = visualize_prediction(img, detections, threshold)

    txt = (
        f"Model: {model_name}\n"
        f"Inference time: {elapsed} s\n"
        f"Detections: {len(detections)}\n\n"
    )

    for d in detections:
        txt += f"- {d['label']}: {d['score']:.3f}  box={tuple(round(x,1) for x in d['box'])}\n"

    return txt, img_out

In [ ]:
def build_classification_interface():
    with gr.Column():
      with gr.Row():
        with gr.Column(scale=1):
          model = gr.Dropdown(list(CLASSIFICATION_MODELS.keys()), label="Model")
          image = gr.Image(type="filepath", label="Image",height=600)
          labels = gr.Textbox(label="Candidate labels", value="")
          run_btn = gr.Button("Run")

        with gr.Column(scale=1):
          text_out = gr.Textbox(label="Result", lines=15)

        run_btn.click(run_classification, inputs=[model, image, labels], outputs=text_out)


def build_detection_interface():
    with gr.Column():
      with gr.Row():
        with gr.Column(scale=1):
          model = gr.Dropdown(list(DETECTION_MODELS.keys()), label="Model")
          image = gr.Image(type="filepath", label="Image", height=600)
          labels = gr.Textbox(label="Candidate labels", value="cat, dog, person")
          threshold = gr.Slider(0.0, 1.0, value=0.3, step=0.01, label="Threshold")
          run_btn = gr.Button("Run")
        with gr.Column(scale=1):
          text_out = gr.Textbox(label="Result", lines=15)
          img_out = gr.Image(label="Output image", height=600, type="pil")

        run_btn.click(run_detection, inputs=[model, image, labels, threshold], outputs=[text_out, img_out])



with gr.Blocks(title="Zero-shot Classification & Detection") as demo:
    gr.Markdown("## Zero-shot Classification & Detection Demo")

    task = gr.Dropdown(choices=["classification", "detection"], label="Task", value="classification")

    classification_ui = gr.Group(visible=True)
    with classification_ui:
        build_classification_interface()

    detection_ui = gr.Group(visible=False)
    with detection_ui:
        build_detection_interface()


    def swap_ui(task_name):
        return (
            gr.update(visible=task_name == "classification"),
            gr.update(visible=task_name == "detection")
        )

    task.change(swap_ui, task, outputs=[classification_ui, detection_ui])

In [ ]:
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b5342d3d009141e7d2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
